In [ ]:
# 전처리
- 사용 데이터: Roboflow construction safety

- 이미지 크기: 640x640

- 클래스: helmet,no-helmet, no-vest, person, vest

- YOLOv8 학습을 위해 data.yaml 생성
- Faster R-CNN 학습을 위해 YOLO txt 라벨을 COCO json 형식으로 변환


In [ ]:
#데이터 경로 확인
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/기계학습_2조")
RAW_DATASET = ROOT / "data" / "raw" / "helmet_dataset"

print("ROOT 존재:", ROOT.exists())
print("RAW_DATASET 존재:", RAW_DATASET.exists())

print("data.yaml 존재:", (RAW_DATASET / "data.yaml").exists())
print("train/images 존재:", (RAW_DATASET / "train" / "images").exists())
print("train/labels 존재:", (RAW_DATASET / "train" / "labels").exists())
print("valid/images 존재:", (RAW_DATASET / "valid" / "images").exists())
print("valid/labels 존재:", (RAW_DATASET / "valid" / "labels").exists())
print("test/images 존재:", (RAW_DATASET / "test" / "images").exists())
print("test/labels 존재:", (RAW_DATASET / "test" / "labels").exists())

ROOT 존재: True
RAW_DATASET 존재: True
data.yaml 존재: True
train/images 존재: True
train/labels 존재: True
valid/images 존재: True
valid/labels 존재: True
test/images 존재: True
test/labels 존재: True


In [ ]:
#클래스 확인
with open(RAW_DATASET / "data.yaml", "r", encoding="utf-8") as f:
    print(f.read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 5
names: ['helmet', 'no-helmet', 'no-vest', 'person', 'vest']

roboflow:
  workspace: roboflow-100
  project: construction-safety-gsnvb
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/roboflow-100/construction-safety-gsnvb/dataset/1


In [ ]:
#전처리 결과 저장 폴더
import shutil
from PIL import Image

PROCESSED_DIR = ROOT / "data" / "processed" / "Detection"

IMAGE_TRAIN_DIR = PROCESSED_DIR / "images" / "train"
IMAGE_VAL_DIR = PROCESSED_DIR / "images" / "val"
IMAGE_TEST_DIR = PROCESSED_DIR / "images" / "test"

LABEL_TRAIN_DIR = PROCESSED_DIR / "yolo_labels" / "train"
LABEL_VAL_DIR = PROCESSED_DIR / "yolo_labels" / "val"
LABEL_TEST_DIR = PROCESSED_DIR / "yolo_labels" / "test"

folders = [
    IMAGE_TRAIN_DIR,
    IMAGE_VAL_DIR,
    IMAGE_TEST_DIR,
    LABEL_TRAIN_DIR,
    LABEL_VAL_DIR,
    LABEL_TEST_DIR,
    PROCESSED_DIR / "rcnn_labels"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("전처리 결과 폴더 생성 완료")

전처리 결과 폴더 생성 완료


In [ ]:
#이미지, 라벨 정리
def resize_image_640(src_path, dst_path):
    img = Image.open(src_path).convert("RGB")
    img = img.resize((640, 640))
    img.save(dst_path)


def process_split(src_split_name, dst_split_name):
    src_image_dir = RAW_DATASET / src_split_name / "images"
    src_label_dir = RAW_DATASET / src_split_name / "labels"

    if dst_split_name == "train":
        dst_image_dir = IMAGE_TRAIN_DIR
        dst_label_dir = LABEL_TRAIN_DIR
    elif dst_split_name == "val":
        dst_image_dir = IMAGE_VAL_DIR
        dst_label_dir = LABEL_VAL_DIR
    else:
        dst_image_dir = IMAGE_TEST_DIR
        dst_label_dir = LABEL_TEST_DIR

    image_extensions = [".jpg", ".jpeg", ".png"]

    image_files = [
        p for p in src_image_dir.iterdir()
        if p.suffix.lower() in image_extensions
    ]

    copied_count = 0
    missing_label_count = 0

    for img_path in image_files:
        label_path = src_label_dir / f"{img_path.stem}.txt"

        dst_img_path = dst_image_dir / img_path.name
        dst_label_path = dst_label_dir / f"{img_path.stem}.txt"

        resize_image_640(img_path, dst_img_path)

        if label_path.exists():
            shutil.copy(label_path, dst_label_path)
            copied_count += 1
        else:
            missing_label_count += 1
            print("라벨 없음:", img_path.name)

    print(f"{src_split_name} → {dst_split_name} 처리 완료")
    print("이미지 처리 수:", len(image_files))
    print("라벨 복사 수:", copied_count)
    print("라벨 없는 이미지 수:", missing_label_count)
    print()

In [ ]:
process_split("train", "train")
process_split("valid", "val")
process_split("test", "test")

train → train 처리 완료
이미지 처리 수: 997
라벨 복사 수: 997
라벨 없는 이미지 수: 0

valid → val 처리 완료
이미지 처리 수: 119
라벨 복사 수: 119
라벨 없는 이미지 수: 0

test → test 처리 완료
이미지 처리 수: 90
라벨 복사 수: 90
라벨 없는 이미지 수: 0



In [ ]:
print("train images:", len(list(IMAGE_TRAIN_DIR.glob("*"))))
print("train labels:", len(list(LABEL_TRAIN_DIR.glob("*"))))

print("val images:", len(list(IMAGE_VAL_DIR.glob("*"))))
print("val labels:", len(list(LABEL_VAL_DIR.glob("*"))))

print("test images:", len(list(IMAGE_TEST_DIR.glob("*"))))
print("test labels:", len(list(LABEL_TEST_DIR.glob("*"))))

train images: 997
train labels: 997
val images: 119
val labels: 119
test images: 90
test labels: 90


In [ ]:
#YOLO 학습용 data.yaml 생성
yaml_text = f"""
train: {IMAGE_TRAIN_DIR}
val: {IMAGE_VAL_DIR}
test: {IMAGE_TEST_DIR}

nc: 5
names: ['helmet', 'vest', 'person', 'no-helmet', 'no-vest']
"""

yaml_path = PROCESSED_DIR / "data.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_text)

print("data.yaml 생성 완료")
print(yaml_path)
print()
print(yaml_text)

data.yaml 생성 완료
/content/drive/MyDrive/기계학습_2조/data/processed/Detection/data.yaml


train: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/images/train
val: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/images/val
test: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/images/test

nc: 5
names: ['helmet', 'vest', 'person', 'no-helmet', 'no-vest']



In [ ]:
# Faster R-CNN용 COCO json 변환

from PIL import Image
import json

IMAGE_ROOT = PROCESSED_DIR / "images"
YOLO_LABEL_ROOT = PROCESSED_DIR / "yolo_labels"

RCNN_LABEL_DIR = PROCESSED_DIR / "rcnn_labels"
RCNN_LABEL_DIR.mkdir(parents=True, exist_ok=True)

print("이미지 폴더:", IMAGE_ROOT)
print("YOLO 라벨 폴더:", YOLO_LABEL_ROOT)
print("R-CNN 라벨 저장 폴더:", RCNN_LABEL_DIR)

print("images 존재:", IMAGE_ROOT.exists())
print("yolo_labels 존재:", YOLO_LABEL_ROOT.exists())
print("rcnn_labels 존재:", RCNN_LABEL_DIR.exists())

이미지 폴더: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/images
YOLO 라벨 폴더: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/yolo_labels
R-CNN 라벨 저장 폴더: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/rcnn_labels
images 존재: True
yolo_labels 존재: True
rcnn_labels 존재: True


In [ ]:
# 클래스 정보
# 기존 데이터셋 클래스 5개를 그대로 유지

categories = [
    {"id": 0, "name": "helmet"},
    {"id": 1, "name": "no-helmet"},
    {"id": 2, "name": "no-vest"},
    {"id": 3, "name": "person"},
    {"id": 4, "name": "vest"}
]

print(categories)

[{'id': 0, 'name': 'helmet'}, {'id': 1, 'name': 'no-helmet'}, {'id': 2, 'name': 'no-vest'}, {'id': 3, 'name': 'person'}, {'id': 4, 'name': 'vest'}]


In [ ]:
# YOLO txt -> COCO json 변환

def make_coco_json(split):
    image_folder = IMAGE_ROOT / split
    label_folder = YOLO_LABEL_ROOT / split

# COCO json 기본 구조 만들기
    coco = {
        "images": [],
        "annotations": [],
        "categories": categories
    }

    image_id = 1
    annotation_id = 1

    image_files = []
    image_files += list(image_folder.glob("*.jpg"))
    image_files += list(image_folder.glob("*.jpeg"))
    image_files += list(image_folder.glob("*.png"))

    image_files = sorted(image_files)

    for img_path in image_files:
        img = Image.open(img_path)
        width, height = img.size

        image_info = {
            "id": image_id,
            "file_name": img_path.name,
            "width": width,
            "height": height
        }

        coco["images"].append(image_info)

        label_path = label_folder / (img_path.stem + ".txt")

        if label_path.exists():
            with open(label_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            for line in lines:
                data = line.strip().split()

                if len(data) != 5:
                    continue

                class_id = int(data[0])
                x_center = float(data[1])
                y_center = float(data[2])
                box_w = float(data[3])
                box_h = float(data[4])

                # YOLO 좌표는 0~1 사이 비율이므로 실제 픽셀 좌표로 변환
                box_w_pixel = box_w * width
                box_h_pixel = box_h * height

                x_min = (x_center * width) - (box_w_pixel / 2)
                y_min = (y_center * height) - (box_h_pixel / 2)

                annotation = {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": class_id,
                    "bbox": [x_min, y_min, box_w_pixel, box_h_pixel],
                    "area": box_w_pixel * box_h_pixel,
                    "iscrowd": 0
                }

                coco["annotations"].append(annotation)
                annotation_id += 1

        image_id += 1

    save_path = RCNN_LABEL_DIR / (split + ".json")

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=4)

    print(split, "변환 완료")
    print("이미지 수:", len(coco["images"]))
    print("라벨 수:", len(coco["annotations"]))
    print("저장 위치:", save_path)
    print()

In [ ]:
make_coco_json("train")
make_coco_json("val")
make_coco_json("test")

train 변환 완료
이미지 수: 997
라벨 수: 6386
저장 위치: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/rcnn_labels/train.json

val 변환 완료
이미지 수: 119
라벨 수: 715
저장 위치: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/rcnn_labels/val.json

test 변환 완료
이미지 수: 90
라벨 수: 623
저장 위치: /content/drive/MyDrive/기계학습_2조/data/processed/Detection/rcnn_labels/test.json



In [ ]:
print("train.json:", (RCNN_LABEL_DIR / "train.json").exists())
print("val.json:", (RCNN_LABEL_DIR / "val.json").exists())
print("test.json:", (RCNN_LABEL_DIR / "test.json").exists())

train.json: True
val.json: True
test.json: True
